# NCM research run: `e1b_continue`

E1b (pre-registered, eval-only, no training; criterion (c) candidate C1): continued E1 sampling at cut 8 on the censored and wide items of the released models (WMT14 J0 and TinyStories A0; the item lists are recomputed from the saved E1 results). Replays E1's streams, then rounds of 512 samples until the item's interval is no wider than 2 nats or 2^15 samples. Needs the released WMT14 and TinyStories checkpoints and the WMT14 data, as e1_equivalence. fp32, TF32 off. Cost: ~60-100 L4-minutes, cap 150 L4-minutes (checked between items; items not reached keep E1's counts).

- Recommended GPU: **L4**. Estimated cost: **6.0 compute units** (cap: 150 L4-minutes).
- Runtime > Change runtime type > GPU, then Runtime > Run all.
- If Colab disconnects, just Run all again: every command resumes from its last Drive checkpoint.
- When done, copy `MyDrive/ncm-research/results/e1b_continue/` into the repo's `results/colab/`.

In [ ]:
import os, subprocess, sys, json, shutil, time
T_START = time.time()
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/ncm-research'
R = f'{DRIVE}/runs'; D = f'{DRIVE}/data/tinystories'
os.makedirs(R, exist_ok=True); os.makedirs(D, exist_ok=True)
REPO, BRANCH = 'https://github.com/amritisepic/NCM-research.git', 'claude/ncm-research'
CMDS = [
 "python scripts/e1b_continue.py --run_dir {R}/e1b --data {R}/../data/wmt14_train_pairs.npz --max_minutes 150"
]
SEEDS = [0]   # edit (e.g. [1, 2]) to run more seeds of the same commands
NEEDS = ["ts", "ckpt", "wmt14"]
NEED_TS = 'ts' in NEEDS or any('scripts/train.py ' in c for c in CMDS)
if not os.path.exists('/content/NCM-research'):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO, '/content/NCM-research'], check=True)
else:
    subprocess.run(f'cd /content/NCM-research && git fetch origin {BRANCH} && git checkout {BRANCH} && git reset --hard origin/{BRANCH}', shell=True, check=True)
os.chdir('/content/NCM-research')
print(subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
GIT_SHA = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'omegaconf', 'hydra-core', 'regex', 'mauve-text', 'faiss-cpu'], check=True)
utc = lambda: time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

def provenance(rd, rec):   # one line per session and command: which code ran, when, on what, from which state
    os.makedirs(rd, exist_ok=True)
    with open(f'{rd}/provenance.jsonl', 'a') as f:
        f.write(json.dumps(rec) + '\n')

def history(rd):
    p = f'{rd}/provenance.jsonl'
    return [json.loads(l) for l in open(p)] if os.path.exists(p) else []

def last_step(rd):
    p, s = f'{rd}/metrics.jsonl', None
    if os.path.exists(p):
        for l in open(p):
            try:
                s = json.loads(l).get('step', s)
            except ValueError:
                pass
    return s

In [ ]:
import urllib.request
os.makedirs('data', exist_ok=True)
if NEED_TS:
    B = 'https://huggingface.co/datasets/hazyresearch/ncm-datasets/resolve/main/tinystories/'
    for f in ['bpe_vocab_8192.pickle', 'bpe_merges_8192.pickle', 'valid_bpe_8192.pickle',
              'train_bpe_8192_offsets.npy', 'train_bpe_8192.bin']:
        p = f'{D}/{f}'
        if not os.path.exists(p):
            print('downloading', f, flush=True)
            urllib.request.urlretrieve(B + f, p + '.tmp'); os.replace(p + '.tmp', p)
    if not os.path.exists('data/tinystories'):
        os.symlink(D, 'data/tinystories')
    # local copy of the big stream for fast random access (Drive reads are slow)
    if not os.path.exists('/content/ts'):
        os.makedirs('/content/ts')
        for f in os.listdir(D):
            shutil.copy(f'{D}/{f}', f'/content/ts/{f}')
if 'ckpt' in NEEDS or 'wmt14' in NEEDS:
    ART = f'{DRIVE}/artifacts'; os.makedirs(ART, exist_ok=True)   # released checkpoints, cached on Drive
    if not os.path.exists('artifacts'):
        os.symlink(ART, 'artifacts')
    subprocess.run('bash scripts/fetch_assets.sh checkpoints', shell=True, check=True)
    if 'wmt14' in NEEDS:
        subprocess.run('bash scripts/fetch_assets.sh wmt14', shell=True, check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sacrebleu'], check=True)
S = 'https://huggingface.co/datasets/hazyresearch/ncm-datasets/resolve/main/sudoku/'
for lvl in ['easy', 'medium', 'hard']:
    os.makedirs(f'data/sudoku/{lvl}', exist_ok=True)
    for f in ['puzzles.pickle', 'solutions.pickle']:
        if not os.path.exists(f'data/sudoku/{lvl}/{f}'):
            urllib.request.urlretrieve(f'{S}{lvl}/{f}', f'data/sudoku/{lvl}/{f}')
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
ALLOW_CODE_CHANGE = False   # True only on purpose: lets a run directory resume under different code than it started with
CODE_PATHS = ['src', 'scripts', 'configs', 'assets']
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
CMDS_RAW = list(CMDS)
for seed in SEEDS:
    for i, c in enumerate(CMDS):
        c = c.format(R=R, D=D, S=seed)
        if 'scripts/train.py ' in c or 'scripts/train_gen.py ' in c:
            c += ' --data_dir /content/ts'
        if 'configs/scale/L_ts_' in c and '--micro_batch' not in c and torch.cuda.is_available() \
                and torch.cuda.get_device_properties(0).total_memory > 39e9:
            c += ' --micro_batch 64'   # 40-80 GB GPUs: same optimizer step, fewer accumulation chunks (13.7 GB at 32)
        if '{S}' not in CMDS_RAW[i] and seed != SEEDS[0]:
            continue   # seed-independent command: run once
        rd = c.split('--run_dir ')[1].split()[0] if '--run_dir ' in c else None
        if rd:
            started = [h['git_sha'] for h in history(rd) if h.get('event') == 'start']
            if not started and os.path.exists(f'{rd}/ckpt.pt') and not ALLOW_CODE_CHANGE:
                raise SystemExit(f'{rd} holds a checkpoint written before provenance was recorded, so the code that trained '
                                 'it is unknown. Rename the run directory to start fresh, or set ALLOW_CODE_CHANGE = True '
                                 'only after checking which commit trained it.')
            # any earlier session may have written the checkpoint: every one of them must have run today's code
            for old in dict.fromkeys(s for s in started if s != GIT_SHA):
                if ALLOW_CODE_CHANGE:
                    break
                if subprocess.run(['git', 'diff', '--quiet', old, GIT_SHA, '--'] + CODE_PATHS).returncode != 0:
                    raise SystemExit(f'{rd} was trained in a session at commit {old[:9]}; the code under {CODE_PATHS} differs '
                                     f'from now ({GIT_SHA[:9]}). Resuming would mix code versions: rename the run directory '
                                     'to start fresh, or set ALLOW_CODE_CHANGE = True only if you mean it.')
            # the command as a template ({R} = the Drive run root) plus the flags this cell added: no absolute path is
            # recorded, because provenance is published with the results (Codex PR 7)
            provenance(rd, {'event': 'start', 'git_sha': GIT_SHA, 'utc': utc(), 'gpu': GPU, 'cmd': CMDS_RAW[i],
                            'seed': seed, 'local_data_copy': ' --data_dir /content/ts' in c,
                            'micro_batch_64': ' --micro_batch 64' in c,
                            'had_ckpt': os.path.exists(f'{rd}/ckpt.pt'), 'last_logged_step': last_step(rd)})
        print('>>>', c, flush=True)
        t = time.time()
        p = subprocess.run(c, shell=True)
        print(f'<<< exit {p.returncode} after {(time.time()-t)/60:.1f} min', flush=True)
        if rd:
            provenance(rd, {'event': 'end', 'git_sha': GIT_SHA, 'utc': utc(), 'exit': p.returncode,
                            'minutes': round((time.time() - t) / 60, 2), 'last_logged_step': last_step(rd)})
        if p.returncode != 0:
            raise SystemExit('command failed; fix and re-run this cell (it resumes)')

In [ ]:
out = f'{DRIVE}/results/e1b_continue'; os.makedirs(out, exist_ok=True)
sessions = {}
for seed in SEEDS:
    for c in CMDS:
        if '--run_dir ' not in c:
            continue
        rd = c.split('--run_dir ')[1].split()[0].format(R=R, S=seed)
        name = os.path.basename(rd)
        sessions[name] = history(rd)
        for f in ['result.json', 'metrics.jsonl', 'config.yaml', 'calib.json', 'hyp_test.json', 'hyp_val.json', 'run_manifest.json', 'eval_final.json', 'eval_6000.json', 'eval_20500.json', 'samples_final.json', 'provenance.jsonl'] + sorted(n for n in (os.listdir(rd) if os.path.isdir(rd) else []) if n.startswith('hyp_val_step') and n.endswith('.json')):
            if os.path.exists(f'{rd}/{f}'):
                shutil.copy(f'{rd}/{f}', f'{out}/{name}__{f}')
for f in os.listdir('results'):   # diagnostics write straight into the repo's results/
    if f.startswith(('diag_', 'b2_jacobi_', 'sudoku_', 'wmt14_')) and os.path.getmtime(f'results/{f}') > T_START:
        shutil.copy(f'results/{f}', f'{out}/{f}')
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
shas = sorted({h['git_sha'] for hs in sessions.values() for h in hs if h.get('git_sha')})
json.dump({'run': 'e1b_continue', 'gpu': gpu, 'finished_utc': utc(), 'git_sha': GIT_SHA, 'git_shas_all_sessions': shas,
           'sessions': sessions}, open(f'{out}/_meta.json', 'w'), indent=1)
print('Results copied to', out, '-> please drop this folder into the repo under results/colab/')
if len(shas) > 1:
    print('NOTE: this run spans several commits:', shas)

In [ ]:
RELEASE_RUNTIME = True   # set False to keep the runtime (e.g. to inspect files)
if RELEASE_RUNTIME:
    from google.colab import runtime
    runtime.unassign()   # results are already copied to Drive by the previous cell